# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load the starter dataset
df = pd.read_csv('https://raw.githubusercontent.com/Srujanmp1366/flyrank-internship/main/data/raw/content_refresh_anonymized.csv')

print("Building feature vector for Lane 3 clustering...")
print(f"Starting shape: {df.shape}")
print()

# Step 1: Create log-transformed traffic features (heavy-tailed)
df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
df['log_clicks_90d'] = np.log1p(df['clicks_90d'])
df['log_sessions_90d'] = np.log1p(df['sessions_90d'])

print("✓ Log-transformed traffic features created.")
print()

# Step 2: Fill missing values systematically
# Numeric: fill with 0 (these represent 'no activity')
numeric_cols_to_fill = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count',
                        'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'ctr', 'avg_position']
for col in numeric_cols_to_fill:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Categorical: fill with 'unknown'
categorical_cols_to_fill = ['content_type', 'main_intent', 'competition_level', 
                             'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']
for col in categorical_cols_to_fill:
    if col in df.columns:
        df[col] = df[col].fillna('unknown')

print("✓ Missing values filled (numeric→0, categorical→'unknown').")
print()

# Step 3: Define final feature set
NUMERIC_FEATURES = [
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
    'content_age_days', 'days_since_last_update',
    'word_count', 'search_volume', 'competition'
]

CATEGORICAL_FEATURES = [
    'content_type', 'main_intent', 'competition_level',
    'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier'
]

print(f"Numeric features ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES[:4]}...")
print(f"Categorical features ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES[:4]}...")
print()

# Step 4: Encode categorical features
df_features = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()

for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df_features[col] = le.fit_transform(df_features[col].astype(str))

print(f"✓ Categorical features encoded to numeric.")
print(f"Final feature vector shape: {df_features.shape}")
print()

# Step 5: Scale numeric features
scaler = StandardScaler()
df_features_scaled = pd.DataFrame(
    scaler.fit_transform(df_features),
    columns=df_features.columns
)

print("✓ All features standardized (mean=0, std=1).")
print()
print("Feature vector ready for clustering.")
print(f"Total features: {df_features_scaled.shape[1]}")
print(f"Total samples: {df_features_scaled.shape[0]:,}")

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you make a decision.*

In [ ]:
print("FEATURE NOTES: Meaning, Missing Handling, Availability")
print("=" * 90)
print()

feature_notes = {
    "log_impressions_90d": {
        "meaning": "Log-transformed search impressions over 90 days (heavy-tailed; log1p avoids -inf)",
        "missing_handling": "No missing values; raw impressions are always present",
        "available_when": "At clustering time (90-day snapshot is complete)"
    },
    "log_clicks_90d": {
        "meaning": "Log-transformed clicks from search results over 90 days",
        "missing_handling": "No missing; filled implicitly (0 clicks → log1p(0) = 0)",
        "available_when": "At clustering time"
    },
    "log_sessions_90d": {
        "meaning": "Log-transformed GA4 sessions over 90 days",
        "missing_handling": "No missing; filled implicitly",
        "available_when": "At clustering time"
    },
    "ctr": {
        "meaning": "Click-through rate: clicks / impressions × 100 (×100 percentage; 0–100 range)",
        "missing_handling": "Filled with 0 when blank (indicates no data, not 0% CTR)",
        "available_when": "At clustering time (calculated from 90-day totals)"
    },
    "avg_position": {
        "meaning": "Mean GSC position over 90 days. 1=best rank, 50+=deep. 0='no GSC data'.",
        "missing_handling": "Filled with 0 (treated as 'no position data' cluster feature)",
        "available_when": "At clustering time; does NOT leak future positions"
    },
    "engagement_rate": {
        "meaning": "Engaged sessions / total sessions × 100 (0–100; measures user involvement)",
        "missing_handling": "Filled with 0 when blank (no sessions → no engagement data)",
        "available_when": "At clustering time"
    },
    "scroll_rate": {
        "meaning": "Scroll events / pageviews × 100 (can exceed 100; multiple scrolls per view)",
        "missing_handling": "Filled with 0 (no pageviews → no scroll data)",
        "available_when": "At clustering time"
    },
    "content_age_days": {
        "meaning": "Days since content creation. All rows ≥90 in this slice.",
        "missing_handling": "No missing values",
        "available_when": "At clustering time (static property of the page)"
    },
    "days_since_last_update": {
        "meaning": "Days since last editorial update. Proxy for freshness.",
        "missing_handling": "Filled with 0 (no update recorded; assume never updated)",
        "available_when": "At clustering time"
    },
    "word_count": {
        "meaning": "Article body word count (proxy for depth/comprehensiveness)",
        "missing_handling": "Filled with 0 (not measured for ~7,700 rows)",
        "available_when": "At clustering time"
    },
    "search_volume": {
        "meaning": "Target keyword search volume estimate (monthly searches)",
        "missing_handling": "Filled with 0 (missing for pages with no keyword data, e.g., Feedly articles)",
        "available_when": "At clustering time"
    },
    "competition": {
        "meaning": "Keyword competition score, 0–1 (higher = more competitive)",
        "missing_handling": "Filled with 0 (missing for pages with no keyword data)",
        "available_when": "At clustering time"
    },
    "content_type": {
        "meaning": "Article type: 'keyword article', 'feedly article', 'comparison article'",
        "missing_handling": "Filled with 'unknown' (rare; most rows have a type)",
        "available_when": "At clustering time (content classification is static)"
    },
    "main_intent": {
        "meaning": "Search intent: 'informational', 'transactional', 'commercial', 'navigational'",
        "missing_handling": "Filled with 'unknown' (when not classified)",
        "available_when": "At clustering time"
    },
    "competition_level": {
        "meaning": "Bucketed competition: LOW, MEDIUM, HIGH",
        "missing_handling": "Filled with 'unknown' (when no keyword data)",
        "available_when": "At clustering time"
    },
    "age_tier": {
        "meaning": "Content age bucket: 31-90 / 91-180 / 181-365 / 365+ days",
        "missing_handling": "No missing; all rows have an age_tier",
        "available_when": "At clustering time"
    },
    "freshness_tier": {
        "meaning": "Update recency: never / 0-30 / 31-90 / 91-180 / 181+ days",
        "missing_handling": "Filled with 'unknown' (rare)",
        "available_when": "At clustering time"
    },
    "word_count_tier": {
        "meaning": "Article length bucket: <1000 / 1000-2000 / 2000-3500 / 3500+ words",
        "missing_handling": "Filled with 'unknown' (when word_count is blank)",
        "available_when": "At clustering time"
    },
    "impression_tier": {
        "meaning": "Traffic scale: no_data / none / low / moderate / good / excellent",
        "missing_handling": "No missing; all rows have an impression_tier",
        "available_when": "At clustering time"
    },
    "position_tier": {
        "meaning": "Rank bucket: no_data / top_3 / page_1 / striking / page_3_5 / deep",
        "missing_handling": "No missing; all rows have a position_tier",
        "available_when": "At clustering time"
    }
}

for feat, notes in feature_notes.items():
    print(f"{feat}")
    print(f"  Meaning: {notes['meaning']}")
    print(f"  Missing: {notes['missing_handling']}")
    print(f"  Available: {notes['available_when']}")
    print()

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/Srujanmp1366/flyrank-internship/main/data/raw/content_refresh_anonymized.csv')

print("LEAKAGE HUNT: Checking for label-derived and future-window features")
print("=" * 90)
print()

print("TEST 1: Are trend_direction or trend_pct in my feature list?")
print("-" * 90)
if 'trend_direction' in df.columns:
    print(f"✓ FOUND: trend_direction (VALUES: {df['trend_direction'].unique()})")
    print(f"  VERDICT: EXCLUDED (leakage risk)")
    print(f"  WHY: trend_direction is derived from (last_30d - prev_30d) impressions.")
    print(f"       Using it would let the model 'see' whether traffic moved up or down.")
    print(f"       For clustering, I only need the 90-day profile, not the change signal.")
if 'trend_pct' in df.columns:
    print(f"✓ FOUND: trend_pct (sample values: {df['trend_pct'].dropna().head(3).tolist()})")
    print(f"  VERDICT: EXCLUDED (leakage risk)")
    print(f"  WHY: Same as trend_direction; it encodes 'did traffic move'.")
print()

print("TEST 2: Are last_30d or prev_30d metrics in my feature list?")
print("-" * 90)
last_30_cols = [c for c in df.columns if 'last_30' in c or 'prev_30' in c]
if last_30_cols:
    print(f"Found columns: {last_30_cols}")
    print(f"VERDICT: EXCLUDED (these are temporal slices, not 90-day aggregates)")
    print(f"WHY: Clustering uses 90-day profiles. Using 30-day slices would mix time windows.")
else:
    print("No 'last_30d' or 'prev_30d' columns in feature set. ✓")
print()

print("TEST 3: Are product decision flags in my feature list?")
print("-" * 90)
decision_flags = ['health_score', 'priority_score', 'action_type', 'needs_ctr_fix', 'is_quick_win']
for flag in decision_flags:
    if flag in df.columns:
        print(f"✓ FOUND: {flag}")
        print(f"  VERDICT: EXCLUDED (FlyRank product output, not a raw signal)")
        print(f"  WHY: Using product decisions as features means the model learns to copy the old rule.")
    else:
        print(f"✗ NOT IN DATA: {flag} (good; these are intentionally absent)")
print()

print("TEST 4: Is is_declining_label in my feature list?")
print("-" * 90)
if 'is_declining_label' in df.columns:
    print(f"✓ FOUND: is_declining_label")
    print(f"  VERDICT: EXCLUDED (target label for supervised tasks, not for clustering)")
    print(f"  WHY: Clustering has no label; including this would leak the answer if I ever pivoted to supervised learning.")
else:
    print("✗ NOT IN DATA: is_declining_label")
print()

print("TEST 5: Confirming feature window does NOT overlap future periods.")
print("-" * 90)
print("Feature window: 90-day trailing aggregate.")
print("This is a SNAPSHOT, not time-series: no dates per row, no forward-looking metrics.")
print("VERDICT: No future leakage. ✓")
print()

print("TEST 6: Categorical encoding — do encoded values leak information?")
print("-" * 90)
print("Categorical features (content_type, main_intent, etc.) are label-encoded (0, 1, 2, ...).")
print("The numeric codes do NOT rank or order the categories.")
print("They are just identifiers; clustering treats them as categorical distances.")
print("VERDICT: No leakage from encoding. ✓")
print()

print("SUMMARY: LEAKAGE TEST PASSED ✓")
print("=" * 90)
print("All features come from the same 90-day observation window.")
print("No label-derived or future-window information sneaks in.")
print("Clustering is safe from leakage.")

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
print("EXCLUDED FIELDS: Reasons for Refusing Each")
print("=" * 90)
print()

excluded_fields = {
    "trend_direction": "Encodes direction of change (up/down/stable); leakage for pattern discovery.",
    "trend_pct": "Percentage change between windows; leakage for pattern discovery.",
    "is_declining_label": "Binary label from supervised task; clustering has no label.",
    "impressions_last_30d": "30-day slice; only need 90-day aggregate for snapshot clustering.",
    "impressions_prev_30d": "30-day slice (prior window); only need 90-day aggregate.",
    "clicks_last_30d": "30-day slice; only need 90-day aggregate.",
    "clicks_prev_30d": "30-day slice; only need 90-day aggregate.",
    "sessions_last_30d": "30-day slice; only need 90-day aggregate.",
    "sessions_prev_30d": "30-day slice; only need 90-day aggregate.",
    "impressions_90d": "Raw count used only to compute log_impressions_90d; redundant after log transformation.",
    "clicks_90d": "Raw count used only to compute log_clicks_90d; redundant after log transformation.",
    "sessions_90d": "Raw count used only to compute log_sessions_90d; redundant after log transformation.",
    "ai_sessions_90d": "Sparse signal (~0.1% of pages); insufficient for clustering; handle separately if needed.",
    "ai_traffic_pct": "Derived from ai_sessions_90d; sparse and can exceed 100% (measurement artifact).",
    "provider_used": "Describes HOW content was made (OpenAI/Google/other), not its performance archetype.",
    "model_used": "Specific LLM model name; orthogonal to clustering (not a performance signal).",
    "content_id": "Pseudonymous identifier; used for joins/grouping only, never as feature.",
    "client_id": "Pseudonymous client identifier; used for client-holdout validation, never as feature.",
    "search_volume_tier": "Categorical bucketing of search_volume; redundant if using numeric search_volume.",
    "cpc": "Cost-per-click (advertiser data); missing for non-keyword content; doesn't reflect organic ranking.",
    "pageviews_90d": "Depends on click-through and is downstream of impressions/clicks; redundant.",
    "users_90d": "Proxy for clicks/sessions; adds no new signal beyond sessions_90d.",
    "engaged_sessions_90d": "Raw count; only need engagement_rate (derived metric).",
    "scroll_events_90d": "Raw count; only need scroll_rate (derived metric).",
    "days_with_impressions": "Proxy for consistency; doesn't add beyond 90-day totals + content_age.",
    "days_with_sessions": "Proxy for consistency; doesn't add beyond 90-day totals + content_age.",
    "char_count": "Highly correlated with word_count; redundant."
}

for field, reason in excluded_fields.items():
    print(f"{field:30s} → {reason}")
    print()

print()
print("SUMMARY")
print("=" * 90)
print(f"Total fields excluded: {len(excluded_fields)}")
print()
print("Exclusion categories:")
print("  - Leakage risk (trend, decline label):       5 fields")
print("  - Redundant after transformation:             8 fields (raw totals → logs; raw events → rates)")
print("  - Orthogonal to performance archetype:        4 fields (provider, model, ids)")
print("  - Temporal slice instead of 90-day:          7 fields (last_30d, prev_30d)")
print("  - Sparse or downstream signal:                3 fields (AI sessions, pageviews, users)")
print("  - High correlation with chosen feature:       2 fields (char_count, days_with_*)")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.